1.	Complete the hierarchy: 

1. Shapely geometry
2. GeoSeries
3.  GeoDataFrame 

2.	What type of object is returned by each expression? Write Shapely geometry, GeoSeries, GeoDataFrame, or pandas Series.

parks.geometry

parks.geometry.iloc[0] 

parks.iloc[0] 

parks.iloc[[0]]


GeoSeries, Shapely geometry, panda series, GeoDataFrame

3. Explain why `parks.geometry.iloc[0]` does not behave exactly like `parks.geometry

parks.geometry is a GeoSeries, while .iloc[0] pulls out one element by postions and creates a plain shapely geometry object.

4.	Write code to display all column names in `parks` as a regular Python list. 

list(parks.columns)


5.	Write code to select only the `NAME` and `geometry` columns from `parks`. 

parks[[“NAME”, “geometry]]

6.	Write code to select the first three rows of `parks` while keeping the result as a GeoDataFrame. 

parks.iloc[[0:3]]

7.	Write code to inspect the geometry type of every feature in `parks`. 

parks.geom_type

8.	What does `parks.crs` tell you? 

It tells you the coordinate reference system of the parks GeoDataFrame

9.	Why can one point still produce a complete graph when you call `.plot()`? 

One point can still produce a complete graph because a shapely geometry object contains its own coordinates and geomerty information, so when you call .plot(), matplotlib uses that information to draw the geometry.

10.	Assume `one_park = parks.geometry.iloc[0]`. What library provides the geometry object stored in `one_park`? 

Shapely geometry

stops.crs # EPSG:4326 

11.	Would EPSG:4326 be a good CRS for measuring distance in feet? Explain briefly.

No because its coordinate system measures latitude and longitufe in degrees, not feet

12.	Write code to create `stops_nc` by reprojecting `stops` to EPSG:2264. 

stops_nc = stops.to_crs(epsg=2264)

13.	Write code to create an `area_sqft` column in `parks_nc` using polygon geometry area. 

parks["area_sqft"] = parks_nc.geometry.area

14.	Write code to calculate the distance from every stop in `stops_nc` to a single polygon named `park_1`. 

distance_park_1 = stops_nc.geometry.distance(park_1)

stops_nc["distance] = stops_nc.geometry.distance(park_1)

15.	If `park_1` is a polygon, does `.distance(park_1)` measure distance to the polygon centroid? Explain. 

No, it doesn't. It measures the shortest distance between 2 geometries. So the shortest distance from each stop to the polygon's boundary, not to its centroid.


16.	Explain each 

Predicate	Meaning

within	 - it is True if one geometry is completely inside another geometry. If the stops are inside the polygon

contains	- it is Ture if one geometry, a, completely contains another geometry, b. It is the same as geometry b, is within a.

intersects	- it is True if the 2 geometries share any space or boundary, if they overlap, inside, or just touching.

touches	- it is True if the 2 geometries touch at their boundaries but don't overlap.

The all return True or False

17.	In `gpd.sjoin(parks, urban, predicate="within")`, which geometry is being tested as within which? 

It is testing if parks is within urban

18.	A park crosses an urban-area boundary. Would `within` be True? Would `intersects` be True? 

within would be False because part of the park lies outside the urban polygon. Intersects would be True because the park overlaps/touches the urban polygon.

19.	Write code that returns how many stops are inside each polygon in a gpd named `parks` that has three polygons. 

def count_stops(park_polygon):
    inside = stops.geometry.within(park_polygon)   
    return inside.sum()                            

parks["stop_count"] = parks.geometry.apply(count_stops)
print(parks[["NAME", "stop_count"]])

urban_parks = gpd.sjoin(parks, urban, predicate="within") 

20.	What is the purpose of `gpd.sjoin()`? 

It combines 2 GeoDataFrames based on their spatial relationship instead of a common ID field.

21.	Why is a spatial join different from a regular pandas join on a common ID field? 

A pandas join matches rows on equal values in a common column, while a spatial join matches rows on their location and geometry.

22.	Write a spatial join that keeps bus stops that fall inside park polygons. Use `stops` as the left GeoDataFrame and `parks` as the right GeoDataFrame. 

stops_in_parks = gpd.sjoin(
    stops,
    parks,
    predicate ="within"
)

23.	After a point-in-polygon spatial join, how could you count how many bus stops were matched to each park? Describe the idea or write code. 

stops_in_parks["NAME].value_counts()

This would count how many rows share each value

24.	Why can a polygon-to-polygon `intersects` spatial join create multiple output rows if each original layer was one polygon in size? 

Because one polygon can intersect multiple polygons in the other GeoDataFrame


25.	You want to answer: “Which parks are within 500 feet of a bus stop?” Write how would you do this with 
GeoPandas  
a.	using a distance and apply method.  

near_stop = (stops_nc.geometry.distance(park_geom) <= 500).any()

b.	using buffers 

stop_buffers = stops.copy()
stop_buffers["geometry"] = stops.geometry.buffer(500)

parks_500ft = gpd.sjoin(
    parks,
    stop_buffers,
    predicate="intersects"
)
